# Co-Produced Outputs and Output-to-Output Lineage

## What you'll learn

- How to author an operation that emits two output roles in a single step
- How to declare output-to-output lineage with `infer_lineage_from={"outputs": [role]}`
- How to verify the resulting provenance edges in the pipeline graph

**Prerequisites:** [Writing Your First Operation](01-writing-an-operation.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No.

---

## What we'll build

A **DatasetSummarizer** operation that, in a single step, generates random
CSV datasets *and* computes summary statistics for each one. The two
outputs share filename stems, so the framework can pair each metric to its
co-produced dataset automatically.

```
Step 0: DatasetSummarizer
┌─────────────────────────────────┐
│ datasets:  dataset_00000.csv     │  ──┐
│            dataset_00001.csv     │    │  output->output
│            ...                   │    │   edges
│ metrics:   dataset_00000_stats   │  ◄─┘
│            dataset_00001_stats   │
│            ...                   │
└─────────────────────────────────┘
```

The pattern is common: any time an operation emits a primary artifact plus
a derived summary or quality metric, you'll reach for it.

---

## Imports

In [ ]:
from __future__ import annotations

import csv
import os
import random
import statistics
from enum import StrEnum, auto
from typing import Any, ClassVar

from pydantic import BaseModel, Field

from artisan.operations.base import OperationDefinition
from artisan.orchestration import PipelineManager
from artisan.schemas import ArtifactResult
from artisan.schemas.artifact.data import DataArtifact
from artisan.schemas.artifact.metric import MetricArtifact
from artisan.schemas.artifact.types import ArtifactTypes
from artisan.schemas.specs.input_models import ExecuteInput, PostprocessInput
from artisan.schemas.specs.output_spec import OutputSpec
from artisan.utils import tutorial_setup

---

## Declare two output roles

Each output role gets its own `OutputSpec` with its own `infer_lineage_from`
declaration. The pattern that connects the two roles is the value:

| Role | `infer_lineage_from` | Meaning |
|---|---|---|
| `datasets` | `{"inputs": []}` | Generative -- no parents. |
| `metrics` | `{"outputs": ["datasets"]}` | Each metric derives from a `datasets` artifact whose name stem matches the metric's stem. |

The framework matches `dataset_00000_stats.json` to `dataset_00000.csv` by
stripping extensions and longest-prefix lookup. The convention you control
is the **filename stem**: pick metric names that share a prefix with their
source dataset names.

---

## Define the operation

Two output roles, one `execute` that writes CSVs and accumulates stats,
and one `postprocess` that returns both roles in a single `ArtifactResult`.
Notice that we don't pass an explicit `lineage` block -- the framework
infers all the edges from `infer_lineage_from` plus filename stems.

In [ ]:
class DatasetSummarizer(OperationDefinition):
    """Generate CSV datasets and compute per-dataset summary stats in one step."""

    name = "dataset_summarizer"
    description = "Emit CSVs alongside per-CSV summary metrics"

    class OutputRole(StrEnum):
        datasets = auto()
        metrics = auto()

    inputs: ClassVar[dict[str, Any]] = {}

    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.datasets: OutputSpec(
            artifact_type="data",
            infer_lineage_from={"inputs": []},
        ),
        OutputRole.metrics: OutputSpec(
            artifact_type=ArtifactTypes.METRIC,
            infer_lineage_from={"outputs": ["datasets"]},
        ),
    }

    class Params(BaseModel):
        count: int = Field(
            default=3, ge=1, description="Number of CSV files to generate"
        )
        rows_per_file: int = Field(default=10, ge=1, description="Rows per CSV")
        seed: int | None = Field(default=None, description="Random seed")

    params: Params = Params()

    def execute(self, inputs: ExecuteInput) -> dict[str, Any]:
        os.makedirs(inputs.execute_dir, exist_ok=True)
        rng = random.Random(self.params.seed)
        summaries: list[dict[str, Any]] = []

        for i in range(self.params.count):
            stem = f"dataset_{i:05d}"
            csv_path = os.path.join(inputs.execute_dir, f"{stem}.csv")
            scores: list[float] = []
            with open(csv_path, "w", newline="") as fh:
                writer = csv.writer(fh)
                writer.writerow(["id", "score"])
                for row in range(self.params.rows_per_file):
                    score = round(rng.uniform(0.0, 1.0), 4)
                    writer.writerow([row, score])
                    scores.append(score)
            summaries.append(
                {
                    "metric_name": f"{stem}_stats",
                    "mean_score": statistics.mean(scores),
                    "row_count": len(scores),
                }
            )
        return {"summaries": summaries}

    def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
        dataset_drafts = [
            DataArtifact.draft(
                content=open(f, "rb").read(),
                original_name=os.path.basename(f),
                step_number=inputs.step_number,
            )
            for f in inputs.file_outputs
            if f.endswith(".csv")
        ]
        metric_drafts = [
            MetricArtifact.draft(
                content={
                    "mean_score": s["mean_score"],
                    "row_count": s["row_count"],
                },
                original_name=s["metric_name"],
                step_number=inputs.step_number,
            )
            for s in inputs.memory_outputs["summaries"]
        ]
        return ArtifactResult(
            success=True,
            artifacts={"datasets": dataset_drafts, "metrics": metric_drafts},
        )


print(f"Operation '{DatasetSummarizer.name}' ready")
print(f"  Outputs: {list(DatasetSummarizer.outputs.keys())}")

Two things to notice:

- **Stems align by construction.** A dataset named `dataset_00000.csv` and
  a metric named `dataset_00000_stats` both start with `dataset_00000`.
  The framework's stem matcher pairs them by stripping extensions and
  finding the longest common prefix.
- **`postprocess` returns both roles in one `ArtifactResult`.** The
  framework finalizes both batches, then walks the lineage declarations
  and creates the appropriate edges -- input→output for `datasets`
  (no parents in this case, since the role is generative) and
  output→output for `metrics`.

---

## Run a single-step pipeline

Because `DatasetSummarizer` is generative, no upstream `IngestData` step
is needed. One `pipeline.run` call is the whole pipeline.

In [ ]:
env = tutorial_setup("co_produced_outputs")

pipeline = PipelineManager.create(
    name="co_produced_outputs_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

step0 = pipeline.run(
    operation=DatasetSummarizer,
    name="summarize",
    params={"count": 3, "rows_per_file": 8, "seed": 42},
)

summary = pipeline.finalize()
print(f"Pipeline success: {summary['overall_success']}")
print(f"Step 0 produced: {step0.succeeded_count} units")

---

## Inspect both output roles

`inspect_pipeline` shows step-level counts, including artifacts of
different types in the same step.

In [ ]:
from artisan.visualization import inspect_pipeline

inspect_pipeline(env.delta_root)

The `produced` column should report a mix of `data` and `metric` rows.
`inspect_metrics` parses metric artifacts into a flat table:

In [ ]:
from artisan.visualization import inspect_metrics

inspect_metrics(env.delta_root, step_number=0)

---

## Verify the output-to-output edges

The micro provenance graph renders one node per artifact, with arrows
pointing from parents to children. Each metric should sit downstream of
its co-produced dataset.

In [ ]:
from artisan.visualization import build_micro_graph

build_micro_graph(env.delta_root)

Each `dataset_NNNNN.csv` node has an arrow pointing to its corresponding
`dataset_NNNNN_stats` metric -- the output→output edges that
`infer_lineage_from={"outputs": ["datasets"]}` produced.

Downstream `Filter` operations can read these edges to discover
co-produced metrics for any dataset, which is how the
[metrics-and-filtering tutorial](../02-pipeline-design/03-metrics-and-filtering.ipynb)
filters datasets by quality without an explicit join.

---

## When stems don't line up

Auto-inference depends on filenames sharing stems. If your operation
produces names like `summary.json` for a dataset named
`dataset_00000.csv`, stem matching has nothing to latch onto. In that
case, declare lineage explicitly by passing a `lineage` block on
`ArtifactResult`. The recipe lives in
[Writing Creator Operations](../../how-to-guides/writing-creator-operations.md#explicit-lineage),
and the underlying mechanism is described in
[Provenance System](../../concepts/provenance-system.md#explicit-lineage).

---

## Summary

You authored an operation that emits two output roles in a single step:

1. **Declared** a generative role (`datasets`) and a derived role
   (`metrics`) on the same operation.
2. **Linked them** with `infer_lineage_from={"outputs": ["datasets"]}`,
   relying on shared filename stems for automatic pairing.
3. **Returned both roles** in one `ArtifactResult` from `postprocess`.
4. **Verified** the output→output edges with `build_micro_graph`.

## Next steps

- [Metrics and Filtering](../02-pipeline-design/03-metrics-and-filtering.ipynb) -- Use co-produced metrics to drive `Filter` decisions.
- [Writing Creator Operations](../../how-to-guides/writing-creator-operations.md) -- Recipes for advanced patterns, including explicit lineage when stems don't align.
- [Provenance System](../../concepts/provenance-system.md) -- The full lineage model and edge resolution pipeline.
- [Glossary](../../reference/glossary.md) -- Definitions for `LineageMapping`, role, stem matching, and related terms.